# Train the last layer

## Imports 

In [3]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "C:\\Users\\sysadmin\\Documents\\stage\\Edge AI\\KWS\\52-Hertz-KWS\\src")))

In [4]:
import tensorflow as tf
import numpy as np
import os, glob
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from preprocessing import AudioPipeline
from preprocessing.dataset_pipeline import load_and_preprocess, make_dataset
from preprocessing.transfer_model import build_transfer_model
from sklearn.metrics import classification_report, confusion_matrix

## Config

In [5]:
DATA_DIR    = "C:\\Users\\sysadmin\\Documents\\stage\\Edge AI\\KWS\\52-Hertz-KWS\\data\\raw"
CLASSES     = ["talky", "silence", "unknown"]
BATCH_SIZE  = 16       # 64 trop grand pour <500 samples (config à ignorer)
EPOCHS_P1   = 20       # backbone gelé
LR_P1       = 1e-3
KERAS_PATH  = "C:\\Users\\sysadmin\\Documents\\stage\\Edge AI\\KWS\\52-Hertz-KWS\\src\\models\\miniresnetv1_s1_64x50_tl.keras"

pipeline = AudioPipeline("C:\\Users\\sysadmin\\Documents\\stage\\Edge AI\\KWS\\52-Hertz-KWS\\configuration\\config.yaml")

## Chargement des chemins + labels 

In [6]:
all_paths, all_labels = [], []

for label_idx, class_name in enumerate(CLASSES):
    wavs = glob.glob(os.path.join(DATA_DIR, class_name, "*.wav"))
    all_paths  += wavs
    all_labels += [label_idx] * len(wavs)
    print(f"  {class_name:10s} → {len(wavs)} fichiers")

all_paths  = np.array(all_paths)
all_labels = np.array(all_labels)
print(f"\nTotal : {len(all_paths)} samples\n")

  talky      → 295 fichiers
  silence    → 135 fichiers
  unknown    → 326 fichiers

Total : 756 samples



## Split 

In [7]:
# train 70% / val 15% / test 15%
X_train, X_temp, y_train, y_temp = train_test_split(
    all_paths, all_labels,
    test_size=0.30,
    stratify=all_labels,    
    random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print(f"Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")


Train : 529 | Val : 113 | Test : 114


## Class weights 

In [8]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))
print(f"\nClass weights : {class_weight_dict}")


Class weights : {0: 0.8559870550161812, 1: 1.856140350877193, 2: 0.7733918128654971}


## tf.data pipeline 

In [10]:
train_ds = make_dataset(X_train, y_train, BATCH_SIZE, augment=True)
val_ds   = make_dataset(X_val,   y_val, BATCH_SIZE,  augment=False)
test_ds  = make_dataset(X_test,  y_test, BATCH_SIZE, augment=False)

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: non-default argument follows default argument (<unknown>, line 17)
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: non-default argument follows default argument (<unknown>, line 17)
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: non-default argument follows default argument (<unknown>, line 17)
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert
Please report th

## model 

In [11]:

model = build_transfer_model(KERAS_PATH, num_classes=3)
model.compile(
    optimizer=tf.keras.optimizers.Adam(LR_P1),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()

Model: "miniresnet_kws"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_1 (InputLayer)          │ (None, 64, 50, 1)         │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_pad (ZeroPadding2D)     │ (None, 70, 56, 1)         │               0 │ input_1[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_conv (Conv2D)           │ (None, 32, 25, 64)        │           3,200 │ conv1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_bn (BatchNormalization) │ (None, 32, 25, 64)        │             256 │ conv1_conv[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1_relu (Activation)       │ (None, 32, 25, 64)        │               0 │ conv1_bn[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pad (ZeroPadding2D)     │ (None, 34, 27, 64)        │               0 │ conv1_relu[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ pool1_pool (MaxPooling2D)     │ (None, 16, 13, 64)        │               0 │ pool1_pad[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_conv (Conv2D)  │ (None, 8, 7, 64)          │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_bn             │ (None, 8, 7, 64)          │             256 │ conv2_block1_1_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_1_relu           │ (None, 8, 7, 64)          │               0 │ conv2_block1_1_bn[0][0]    │
│ (Activation)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_conv (Conv2D)  │ (None, 8, 7, 64)          │          36,928 │ conv2_block1_1_relu[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_conv (Conv2D)  │ (None, 8, 7, 64)          │           4,160 │ pool1_pool[0][0]           │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_bn             │ (None, 8, 7, 64)          │             256 │ conv2_block1_2_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_0_bn             │ (None, 8, 7, 64)          │             256 │ conv2_block1_0_conv[0][0]  │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2_block1_2_relu           │ (None, 8, 7, 64)          │               0 │ conv2_block1_2_bn[0][0]    │
│ (Activation)                  │                           │               

 Total params: 101,827 (397.76 KB)

 Trainable params: 10,755 (42.01 KB)

 Non-trainable params: 91,072 (355.75 KB)

## Callbacks 

In [12]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_model.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,                  # généreux : courbes bruitées avec peu de data
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-6,
        verbose=1
    ),
    tf.keras.callbacks.TensorBoard(log_dir="./logs")
]


## Training

In [13]:
print("\n Feature extraction (backbone gelé)")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_P1,
    callbacks=callbacks,
    class_weight=class_weight_dict   # compense le déséquilibre
)


 Feature extraction (backbone gelé)
Epoch 1/20
33/34 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.3297 - loss: 1.5142
Epoch 1: val_accuracy improved from None to 0.38938, saving model to best_model.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 14s 70ms/step - accuracy: 0.3573 - loss: 1.4041 - val_accuracy: 0.3894 - val_loss: 1.0986 - learning_rate: 0.0010
Epoch 2/20
31/34 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4748 - loss: 1.1010
Epoch 2: val_accuracy improved from 0.38938 to 0.45133, saving model to best_model.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 0.4839 - loss: 1.0644 - val_accuracy: 0.4513 - val_loss: 0.9473 - learning_rate: 0.0010
Epoch 3/20
32/34 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.5668 - loss: 0.9580
Epoch 3: val_accuracy improved from 0.45133 to 0.52212, saving model to best_model.keras
34/34 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step - accuracy: 0.5784 - loss: 0.9259 - val_accuracy: 0.5221 - val_loss: 0.8228 - learning_rate: 0.0010
Epoch 4/20
26/34 ━━━━━━

## Best model 

In [14]:
model = tf.keras.models.load_model("best_model.keras")  # epoch 17, 90.3% val acc
loss, acc = model.evaluate(test_ds, verbose=0)
print(f"Test accuracy : {acc:.3f}")

Test accuracy : 0.930


## Confusion matrix

In [16]:

y_pred = np.argmax(model.predict(test_ds), axis=1)
y_true = np.concatenate([y for _, y in test_ds], axis=0)
print(classification_report(y_true, y_pred, target_names=["talky","silence","unknown"]))

8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 143ms/step
              precision    recall  f1-score   support

       talky       0.95      0.91      0.93        45
     silence       0.80      1.00      0.89        20
     unknown       0.98      0.92      0.95        49

    accuracy                           0.93       114
   macro avg       0.91      0.94      0.92       114
weighted avg       0.94      0.93      0.93       114

